# RNA Seq Pipeline
#### Author: Kylie Gregory 
#### Created On: 11/25/2026
#### Last Modified: 11/25/2026
#### Purpose: Create an RNA-seq pipeline to determine what gene expression changes are consistent across two independent datasets comparing dilated cardiomyopathy (DCM) and healthy (non-failing donor) groups. 

#### Files: 


In [2]:
#load in libraries

library(DESeq2)
library(tidyverse)
library(GEOquery)

In [3]:
#read in RNA-seq raw counts dataset


create_raw_counts_table <- function(raw_counts_url, GEO_number){
    #either GSE141910.tsv.gz or GSE116250.tsv.gz
    file_name <- paste(GEO_number, ".tsv.gz", sep="")
    download.file(raw_counts_url, file_name)
    count_table <- read.table(file_name, header=T,sep="\t")
    #make GeneID column the unique row identifiers 
    count_table <- column_to_rownames(count_table, var ="GeneID")

    return(count_table)
    
}

In [4]:

#load in column data to map each sample to a condition 

create_column_data <- function(GEO_number){
                               
    #get the GEO series                          
    gse <- getGEO(GEO_number)
                               
    #get the phenotypic data about each sample                           
    sample_data <- pData(gse[[1]])
    
     #the condition (DCM vs non-failing) is in different columns for the two datasets we are analysing, so created an if/else block 
                               
    #this bloc filters to only DCM vs non-failing samples, selects only that column, changes the name of the column to condition, and edits the exact wording of non-failing and DCM to be consistent among the datasets to allow downstream code to be modular
                              
                 
    if (GEO_number =="GSE141910")
        {
        col_data <- sample_data %>% filter(characteristics_ch1 %in% c("etiology: Non-Failing Donor",
                                                                    "etiology: Dilated cardiomyopathy (DCM)")) %>% select(characteristics_ch1) %>% rename(condition=characteristics_ch1)
        
        col_data$condition <- sub("etiology: ", "", col_data$condition)
            
    } else{
        col_data <-  col_data <- sample_data %>% filter(`disease:ch1` %in% c("non-failing", "dilated cardiomyopathy")) %>% select(`disease:ch1`) %>% rename(condition=`disease:ch1`)
        
        col_data$condition <- sub("non-failing", "Non-Failing Donor", col_data$condition)
        col_data$condition <- sub("dilated cardiomyopathy", "Dilated cardiomyopathy (DCM)", 
                                  col_data$condition)
    
    }
    
    
    return (col_data)

}



#Link for working with GEOquery: https://bioconductor.org/packages/release/bioc/vignettes/GEOquery/inst/doc/GEOquery.html#why-geoquery
                               



In [5]:
d1_geo_number <- "GSE141910"
d2_geo_number <- "GSE116250"

d1_count_url <- "https://www.ncbi.nlm.nih.gov/geo/download/?type=rnaseq_counts&acc=GSE141910&format=file&file=GSE141910_raw_counts_GRCh38.p13_NCBI.tsv.gz"
d2_count_url <- "https://www.ncbi.nlm.nih.gov/geo/download/?type=rnaseq_counts&acc=GSE116250&format=file&file=GSE116250_raw_counts_GRCh38.p13_NCBI.tsv.gz"

#get each datasets raw counts matrix
d1_count_table <- create_raw_counts_table(d1_count_url, d1_geo_number)
d2_count_table <- create_raw_counts_table(d2_count_url, d2_geo_number)

#get each datasets condition information 
d1_col_data <- create_column_data(d1_geo_number)
d2_col_data <- create_column_data(d2_geo_number)
    

Found 1 file(s)



GSE141910_series_matrix.txt.gz



Found 1 file(s)



GSE116250_series_matrix.txt.gz



Using locally cached version of GPL16791 found here:
/tmp/Rtmp83t8uE/GPL16791.soft.gz 



In [6]:
#count matrix has all of the samples, so need to filter to just get DCM and non-failing donor 
#for DESeq2, need rows of col_data to match columns of count_tale


create_filtered_tables <- function(col_data, count_table){
    
    #col_data contains only sample numbers belonging to DCM and non failing donors 
    col_data_samples <- rownames(col_data)
    
    #count_table contains sample numbers from every condition 
    all_samples <- colnames(count_table)
    
    #get overlapping sample numbers 
    dcm_nf_samples <- intersect(col_data_samples, all_samples)
    
    #new count table contains only DCM and nonfailing donors that we also have informatation about in col_data
    count_table_filtered <- count_table %>%
    select(all_of(dcm_nf_samples))
    
    #new col_data contains only DCM and nonfailing donors that
    #were also available in the filtered count table
    
    col_data_filtered <- col_data %>% filter(rownames(col_data) %in% dcm_nf_samples)
    
    tables <- list(count_table_filtered, col_data_filtered)
    
    return (tables)
    
}



In [7]:
d1_final_tables <- create_filtered_tables(d1_col_data, d1_count_table)
d2_final_tables <- create_filtered_tables(d2_col_data, d2_count_table)



d1_final_count_table <- d1_final_tables[[1]]
d2_final_count_table <- d2_final_tables[[1]]


d1_final_col_data  <- d1_final_tables[[2]]
d2_final_col_data  <- d2_final_tables[[2]]


In [8]:
#run DESeq analysis 


run_DESeq <- function(count_table, col_data){
    
    dds <- DESeqDataSetFromMatrix(count_table, col_data, design = ~ condition)
    dds <- DESeq(dds)
    
    return (dds) 
}



In [35]:
#find differentially expressed genes (p_adjusted < 0.05 and log2fold change >=1)


get_DESeq_results <- function(dds){
    
    res <- results(dds,contrast=c("condition","Dilated cardiomyopathy (DCM)", "Non-Failing Donor"), alpha = 0.05, lfcThreshold =1)

    res <- as.data.frame(res) %>% rownames_to_column(var="GeneID")

    all_genes <- res
    
    significant_genes <- res %>% filter(!is.na(res$padj) & padj < 0.05)
    upreg_genes <- significant_genes %>% filter(log2FoldChange > 0) 
    downreg_genes <- significant_genes %>% filter(log2FoldChange < 0) 
  #  downreg_genes <- downreg_genes %>% rownames_to_column(var = "GeneID")
    
    
    genes <- list(upreg_genes, downreg_genes, all_genes)
    
    }
    


In [10]:
d1_dds <- run_DESeq(d1_final_count_table, d1_final_col_data)

d2_dds <- run_DESeq(d2_final_count_table, d2_final_col_data)

Warning message in DESeqDataSet(se, design = design, ignoreRank):
“some variables in design formula are characters, converting to factors”


  Note: levels of factors in the design contain characters other than
  letters, numbers, '_' and '.'. It is recommended (but not required) to use
  only letters, numbers, and delimiters '_' or '.', as these are safe characters
  for column names in R. [This is a message, not a warning or an error]



estimating size factors



  Note: levels of factors in the design contain characters other than
  letters, numbers, '_' and '.'. It is recommended (but not required) to use
  only letters, numbers, and delimiters '_' or '.', as these are safe characters
  for column names in R. [This is a message, not a warning or an error]



estimating dispersions



gene-wise dispersion estimates



... (20 outputs left out)

-- replacing outliers and refitting for 186 genes
-- DESeq argument 'minReplicatesForReplace' = 7 
-- original counts are preserved in counts(dds)



estimating dispersions



fitting model and testing



  Note: levels of factors in the design contain characters other than
  letters, numbers, '_' and '.'. It is recommended (but not required) to use
  only letters, numbers, and delimiters '_' or '.', as these are safe characters
  for column names in R. [This is a message, not a warning or an error]



In [36]:
d1_genes_of_interest <- get_DESeq_results(d1_dds)
d1_up_genes <- d1_genes_of_interest[[1]]
d1_down_genes <- d1_genes_of_interest[[2]]
d1_all_genes <- d1_genes_of_interest[[3]]


d2_genes_of_interest <- get_DESeq_results(d2_dds)
d2_up_genes <- d2_genes_of_interest[[1]]
d2_down_genes <- d2_genes_of_interest[[2]]
d2_all_genes <- d2_genes_of_interest[[3]]

GeneID,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
100133612,6.0551557,-1.512674,0.14054919,-10.762592,1.323276e-04,7.205612e-03
107985467,2.8165421,-1.557096,0.17574726,-8.859858,7.625402e-04,3.461113e-02
5320,1962.0295889,-1.826972,0.17091496,-10.689360,6.541219e-07,5.498434e-05
127731,10.3093757,-3.704266,0.23492083,-15.768145,5.779871e-31,4.882159e-28
400745,3.9335400,-2.084681,0.25981121,-8.023829,1.490713e-05,9.814897e-04
8547,829.9237854,-2.569903,0.11506464,-22.334425,1.101153e-42,2.243244e-39
105378685,6.8362788,-2.319252,0.19542355,-11.867824,7.354978e-12,1.212941e-09
105378699,0.5914063,-1.901940,0.26513538,-7.173467,3.347068e-04,1.645865e-02
1580,1302.7235545,-2.221525,0.12418554,-17.888754,3.927830e-23,1.889286e-20


In [37]:
#Get gene annotations to map gene IDs to their gene symbols 

create_gene_annot_table <- function(gene_annot_file)
    
    {
    
    gene_annot_table <- read.table(gene_annot_file, header=TRUE,
                                   sep = "\t", fill=TRUE, quote="")
    
    gene_annot_table_filtered <- gene_annot_table %>% select(GeneID, Symbol)
    
    gene_annot_table_filtered$GeneID <- as.character(gene_annot_table_filtered$GeneID)
    
    return (gene_annot_table_filtered)
    
}

In [38]:
gene_annot_file <- "Human.GRCh38.p13.annot.tsv.gz"
gene_annot_table <- create_gene_annot_table(gene_annot_file)

head(gene_annot_table)

,GeneID,Symbol
,<chr>,<chr>
1,100287102,DDX11L1
2,653635,WASH7P
3,102466751,MIR6859-1
4,107985730,MIR1302-2HG
5,100302278,MIR1302-2
6,645520,FAM138A


In [39]:
#map gene symbols to geneIDs


create_DEG_symbol_tables <- function(up_genes, down_genes, all_genes, gene_annot_table){
    
    all_genes_with_symbol <- left_join(all_genes, gene_annot_table, by = "GeneID")
    all_genes_with_symbol <- all_genes_with_symbol %>% relocate(Symbol, .before=GeneID)

    up_genes_with_symbol <- left_join(up_genes, gene_annot_table, by = "GeneID")
    up_genes_with_symbol <- column_to_rownames(up_genes_with_symbol, var="Symbol")
   
    
    down_genes_with_symbol <- left_join(down_genes, gene_annot_table, by = "GeneID")
    down_genes_with_symbol <- column_to_rownames(down_genes_with_symbol, var="Symbol")
    
    genes <- list(up_genes_with_symbol, down_genes_with_symbol, all_genes_with_symbol)
    
    return (genes)
    
    
    
}




In [40]:
#get final DEG tables

d1_final_tables <- create_DEG_symbol_tables(d1_up_genes, d1_down_genes, d1_all_genes, gene_annot_table)

d1_final_up_genes_table<- d1_final_tables[[1]]
d1_final_down_genes_table <- d1_final_tables[[2]]
d1_final_all_genes_table <- d1_final_tables[[3]]


d2_final_tables <- create_DEG_symbol_tables(d2_up_genes, d2_down_genes, d2_all_genes, gene_annot_table)

d2_final_up_genes_table <- d2_final_tables[[1]]
d2_final_down_genes_table <- d2_final_tables[[2]]
d2_final_all_genes_table <- d2_final_tables[[3]]


head(d1_final_up_genes_table)

,GeneID,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
LINC02593,100130417,18.497064,2.194921,0.1435808,15.287006,4.314066e-17,1.114961e-14
LOC107985728,107985728,6.686829,1.619084,0.1848504,8.758888,4.053619e-04,1.947086e-02
GABRD,2563,13.091062,1.546212,0.1408494,10.977774,5.266143e-05,3.128251e-03
CHD5,26038,12.730864,1.664715,0.1298902,12.816321,1.547960e-07,1.448470e-05
NPPA-AS1,100379251,9707.713541,2.675511,0.1842482,14.521231,4.783177e-20,1.656510e-17
NPPA,4878,32038.758405,2.771917,0.1952611,14.195951,5.704642e-20,1.956071e-17


In [41]:
#get gene names for Fisher test 

d1_up_gene_names <- rownames(d1_final_up_genes_table)
d2_up_gene_names <- rownames(d2_final_up_genes_table)
d1_all_gene_names <- d1_final_all_genes_table$Symbol

d1_down_gene_names <- rownames(d1_final_down_genes_table)
d2_down_gene_names <- rownames(d2_final_down_genes_table)
d2_all_gene_names <- d2_final_all_genes_table$Symbol

In [57]:
#Fisher test for upregulated genes

all_genes_overlap_fisher <- intersect(d1_all_gene_names, d2_all_gene_names)
d1_up_genes_fisher <- intersect(all_genes_overlap_fisher, d1_up_gene_names)
d2_up_genes_fisher <- intersect(all_genes_overlap_fisher, d2_up_gene_names)

a_up <- length(all_genes_overlap_fisher) - length(union(d1_up_genes_fisher, d2_up_genes_fisher))
b_up <- length(setdiff(d1_up_genes_fisher, d2_up_genes_fisher))
c_up <- length(setdiff(d2_up_genes_fisher, d1_up_genes_fisher))
d_up <- length(intersect(d1_up_genes_fisher, d2_up_genes_fisher))

fisher_up_reg <- fisher.test(matrix(c(a_up, b_up, c_up, d_up), nrow=2), alternative="greater")

fisher_up_reg





	Fisher's Exact Test for Count Data

data:  matrix(c(a_up, b_up, c_up, d_up), nrow = 2)
p-value < 2.2e-16
alternative hypothesis: true odds ratio is greater than 1
95 percent confidence interval:
 138.7031      Inf
sample estimates:
odds ratio 
  207.1376 


In [58]:
#Fisher test for downregulated genes

all_genes_overlap_fisher <- intersect(d1_all_gene_names, d2_all_gene_names)
d1_down_genes_fisher <- intersect(all_genes_overlap_fisher, d1_down_gene_names)
d2_down_genes_fisher <- intersect(all_genes_overlap_fisher, d2_down_gene_names)

a_down <- length(all_genes_overlap_fisher) - length(union(d1_down_genes_fisher, d2_down_genes_fisher))
b_down <- length(setdiff(d1_down_genes_fisher, d2_down_genes_fisher))
c_down <- length(setdiff(d2_down_genes_fisher, d1_down_genes_fisher))
d_down <- length(intersect(d1_down_genes_fisher, d2_down_genes_fisher))

fisher_down_reg <- fisher.test(matrix(c(a_down, b_down, c_down, d_down), nrow=2), alternative="greater")

fisher_down_reg


	Fisher's Exact Test for Count Data

data:  matrix(c(a_down, b_down, c_down, d_down), nrow = 2)
p-value < 2.2e-16
alternative hypothesis: true odds ratio is greater than 1
95 percent confidence interval:
 171.4333      Inf
sample estimates:
odds ratio 
  313.1686 


In [ ]:
#running fisher's exact test 
#documentation for test: https://bioconductor.org/packages/devel/bioc/manuals/GeneOverlap/man/GeneOverlap.pdf

# fisher.test(matrix(c(110, 20, 60, 10), nrow=2), alternative="greater")

# matrix(c(n - union(A,B), setdiff(A,B), setdiff(B,A), intersect(A,B)), nrow=2)

# n = # of genes, A = # genes in set A, B = # of genes in set B

#n - union(A,B) = significant in neither



fisher.test(matrix(c(all_genes_overlap_fisher - union(d1_up_genes_fisher, d2_up_genes_fisher), setdiff(d1_up_genes_fisher, d2_up_genes_fisher), setdiff(d1_up_genes_fisher, d1_up_genes_fisher), intersect(d1_up_genes_fisher, d2_up_genes_fisher)), nrow=2), alternative="greater")

